In [1]:
import requests
import pandas as pd
from geopy.distance import geodesic
import random
from shapely.geometry import Polygon, Point

/Users/maika/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def generate_random_points_with_exclusions(main_coords, exclusion_zones, num_points):
    """
    Generates random (latitude, longitude) coordinates within a main polygon,
    excluding specified sub-areas.

    :param main_coords: List of (lon, lat) tuples for the main area.
    :param exclusion_zones: List of lists of (lon, lat) tuples to exclude.
    :param num_points: Integer, number of valid points to generate.
    :return: List of (latitude, longitude) tuples.
    """
    # 1. Create the main polygon
    valid_area = Polygon(main_coords)

    # 2. Subtract each exclusion zone from the main polygon
    for zone in exclusion_zones:
        exclusion_poly = Polygon(zone)
        valid_area = valid_area.difference(exclusion_poly)

    # 3. Get the bounding box of the new, modified area
    min_x, min_y, max_x, max_y = valid_area.bounds

    valid_points = []

    while len(valid_points) < num_points:
        # Generate point within the bounding box
        random_point = Point(random.uniform(min_x, max_x), random.uniform(min_y, max_y))

        # Check if the modified polygon contains the point
        if valid_area.contains(random_point):
            # Append as (Latitude, Longitude)
            valid_points.append((random_point.y, random_point.x))

    return valid_points

def generate_random_points_sodermalm(N):
    # generates N random points within the Södermalm area, excluding certain zones.
    # returns: list of (latitude, longitude) tuples
    # Main area: A rough box around Södermalm (Lon, Lat)
    sodermalm_polygon = [
        (18.015229, 59.318455), #Reimersholme
        (18.076370, 59.303224), #Eriksdalsbadet
        (18.098269, 59.308377), #Barnängsbryggan
        (18.107320, 59.315768), #Masthamnen
        (18.073617, 59.321215), #Guldbron
        (18.049938, 59.321247)  #Söder mälarstrand (S/S Orion)
    ]

    # Exclusion 1: Tantolunden park (no drone deliveries inside the park)
    tantolunden = [
        (18.0400, 59.3150),
        (18.0550, 59.3150),
        (18.0550, 59.3100),
        (18.0400, 59.3100)
    ]

    # Exclusion 2: A small lake or restricted airspace
    restricted_zone = [
        (18.0700, 59.3100),
        (18.0800, 59.3100),
        (18.0800, 59.3050),
        (18.0700, 59.3050)
    ]

    # Generate 100 points excluding the park and restricted zone
    exclusions = [tantolunden, restricted_zone]
    delivery_locations = generate_random_points_with_exclusions(sodermalm_polygon, exclusions, N)

    return delivery_locations

In [3]:
# First step: Generate 400 random orders from Sodermalm
delivery_locations = generate_random_points_sodermalm(400)
delivery_df = pd.DataFrame({
    "latitude": [d[0] for d in delivery_locations],
    "longitude": [d[1] for d in delivery_locations]
})

In [4]:
def get_route_distance(origin_lat, origin_lon, destination_lats, destination_lons):

    # Restaurant + delivery locations
    coordinates = [[origin_lon, origin_lat]]
    for lat, lon in zip(destination_lats, destination_lons):
        coordinates.append([lon, lat])

    url = "https://api.openrouteservice.org/v2/matrix/cycling-regular"

    with open('../key/ors_api_key.txt', 'r') as f:
        API_KEY = f.read().strip()
    headers = {
        "Authorization": API_KEY,
        "Content-Type": "application/json"
    }

    body = {
        "locations": coordinates,
        "sources": [0],
        "destinations": list(range(1, len(coordinates))),
        "metrics": ["distance"]
    }

    response = requests.post(url, headers = headers, json = body)
    response.raise_for_status()
    result = response.json()

    # ORS returns distances in meters
    distances = [
        d / 1000 if d is not None else None
        for d in result["distances"][0]
    ]

    return distances

In [5]:
# coordinates of max medis
restaurant_lat = 59.31566577019069
restaurant_lon = 18.07306669978915

In [6]:
delivery_df["moped_road_distance"] = get_route_distance(
    restaurant_lat, restaurant_lon, 
    delivery_df["latitude"].tolist(), delivery_df["longitude"].tolist()
)

In [7]:
delivery_df.to_csv("../data/delivery_locations.csv", index=False)

In [9]:
# get geodesic distance
def get_geodesic_distance(origin_lat, origin_lon, destination_lats, destination_lons):
    distances = [
        geodesic((origin_lat, origin_lon), (lat, lon)).kilometers
        for lat, lon in zip(destination_lats, destination_lons)
    ]
    return distances

delivery_df["distance_geodesic_km"] = get_geodesic_distance(
    restaurant_lat, restaurant_lon,
    delivery_df["latitude"].tolist(), delivery_df["longitude"].tolist()
)

In [10]:
# compute moped delivery time
def add_moped_delivery_time(delivery_df):
    # compute moped delivery time in hours
    # time = distance / speed
    # speed = 15 km / h
    delivery_df["moped_delivery_time_m"] = delivery_df["distance_geodesic_km"] / 15 * 60
    return delivery_df

def add_drone_delivery_time(delivery_df):
    # compute drone delivery time in hours
    # time = distance / speed
    # speed = 50 km / h
    delivery_df["drone_delivery_time_m"] = delivery_df["distance_geodesic_km"] / 50 * 60
    return delivery_df

In [11]:
delivery_df = add_moped_delivery_time(delivery_df)
delivery_df = add_drone_delivery_time(delivery_df)


In [12]:
# with unlimited drones, we can achieve an improvement in delivery time.
(delivery_df['moped_delivery_time_m'] - delivery_df['drone_delivery_time_m']).describe()

count    400.000000
mean       3.100375
std        1.728135
min        0.144184
25%        1.722326
50%        2.841131
75%        4.003401
max        9.008754
dtype: float64

In [13]:
# calculate drone unavilability time
def add_drone_unavailability_time(delivery_df):
    speed_drone = 50
    t = delivery_df['distance_geodesic_km'] / speed_drone * 60 * 2 # tur och retur
    # charging time, 25 min per 30 km
    t += delivery_df['distance_geodesic_km'] * (25 / 30)
    delivery_df['drone_unavailability_time'] = t
    return delivery_df

In [14]:
delivery_df = add_drone_unavailability_time(delivery_df)

In [16]:
delivery_df.to_csv("../data/delivery_locations_full.csv", index=False)

In [18]:
delivery_df

,latitude,longitude,moped_road_distance,distance_geodesic_km,moped_delivery_time_m,drone_delivery_time_m,drone_unavailability_time
0,59.317080,18.068377,1.03767,0.310061,1.240244,0.372073,1.002531
1,59.316517,18.048048,2.30605,1.427902,5.711609,1.713483,4.616884
2,59.311686,18.081863,0.98307,0.668945,2.675779,0.802734,2.162922
3,59.304821,18.071387,2.08877,1.211921,4.847684,1.454305,3.918544
4,59.311502,18.076348,0.93945,0.500071,2.000283,0.600085,1.616895
...,...,...,...,...,...,...,...
395,59.320043,18.041132,3.14636,1.882781,7.531126,2.259338,6.087660
396,59.311209,18.069655,1.25656,0.533105,2.132418,0.639725,1.723705
397,59.317702,18.038884,2.93183,1.959787,7.839147,2.351744,6.336644
398,59.318858,18.080999,2.16192,0.574840,2.299360,0.689808,1.858649
